In [ ]:
import os
import glob
import pandas as pd
from pathlib import Path

# Years to process
years = ["2021", "2022"]

# Language groups
LANG = ["eng", "raw", "ar", "vi", "th", "fr", "ru", "he", "sw", "ga", "hi", "zh"]
LANG_FIRST = [f"{l}_first" for l in LANG if l != "raw"]
LANG_LAST = [f"{l}_last" for l in LANG if l != "raw"]
LANG_CRIT = [f"{l}_crit" for l in LANG]

GROUPS = {
    "Base": LANG,
    "First": LANG_FIRST,
    "Last": LANG_LAST
}

ALL_VARIANTS = []
for v in GROUPS.values():
    ALL_VARIANTS.extend(v)


In [53]:
results = []

for year in years:
    base_dir = Path("../../outputs/llm_label") / f"trec_dl_{year}"
    if base_dir.exists():
        models = [f for f in base_dir.iterdir() if f.is_dir()]
        for model_path in models:
            model_name = model_path.name
            
            label_files = glob.glob(str(model_path / "*_labels.csv"))
            
            for file_path in label_files:
                filename = Path(file_path).name
                
                prefix = f"{model_name}_trecdl_{year}_"
                suffix = "_labels.csv"
                
                if filename.startswith(prefix) and filename.endswith(suffix):
                    lang_variant = filename[len(prefix):-len(suffix)]
                else:
                    lang_variant = filename.replace(suffix, "")
                    
                if ALL_VARIANTS and lang_variant not in ALL_VARIANTS:
                    continue
                    
                try:
                    df = pd.read_csv(file_path)
                    if 'llm_relevance' in df.columns and 'relevance' in df.columns:
                        valid = df.dropna(subset=['llm_relevance', 'relevance'])
                        if not valid.empty:
                            mae = (valid['llm_relevance'] - valid['relevance']).abs().mean()
                            mean_diff = (valid['llm_relevance'] - valid['relevance']).mean()
                            rmse = ((valid['llm_relevance'] - valid['relevance'])**2).mean()**0.5
                            
                            group_name = "Other"
                            for g_name, g_list in GROUPS.items():
                                if lang_variant in g_list:
                                    group_name = g_name
                                    break
                                    
                            results.append({
                                'Group': group_name,
                                'Year': str(year),
                                'Model': model_name,
                                'Language/Variant': lang_variant,
                                'MAE': mae,
                                'Mean-Diff': mean_diff,
                                'RMSE': rmse
                            })
                except Exception as e:
                    print(f"Could not process {filename}: {e}")


In [54]:
results_df = pd.DataFrame(results)
if not results_df.empty:
    output_dir = Path("../../outputs/metrics_table")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    for group_name in ["Base", "First", "Last"]:
        group_df_all = results_df[results_df['Group'] == group_name]
        if group_df_all.empty:
            continue
            
        latex_lines = []
        latex_lines.append(r"\begin{table}[t]")
        latex_lines.append(r"\centering")
        latex_lines.append(f"\\caption{{Agreement between LLM-generated relevance labels and NIST judgments for {group_name} variants on TREC-DL 2021 and 2022.}}")
        latex_lines.append(f"\\label{{tab:metrics_{group_name.lower()}}}")
        latex_lines.append(r"\setlength{\tabcolsep}{3.5pt}")
        latex_lines.append(r"\footnotesize")
        
        for year in ["2021", "2022"]:
            group_df = group_df_all[group_df_all['Year'] == year]
            if group_df.empty:
                continue
                
            models = sorted(group_df['Model'].unique())
            metrics = ['MAE', 'Mean-Diff', 'RMSE']
            langs = sorted(group_df['Language/Variant'].unique())
            if 'raw' in langs:
                langs.remove('raw')
                langs.insert(0, 'raw')
            
            col_format = "l" + "c" * (len(models) * len(metrics))
            
            latex_lines.append(f"\\textbf{{TREC-DL {year}}} \\\\")
            latex_lines.append(r"\vspace{0.2em}")
            
            tabular_line = r"\begin{tabular}{" + col_format + r"}"
            latex_lines.append(tabular_line)
            latex_lines.append(r"\toprule")
            
            metric_headers = [""]
            for metric in metrics:
                metric_headers.append(f"\\multicolumn{{{len(models)}}}{{c}}{{\\textbf{{{metric}}}}}")
            latex_lines.append(" & ".join(metric_headers) + " \\\\")
            
            cmidrules = []
            current_col = 2
            for _ in metrics:
                end_col = current_col + len(models) - 1
                cmidrules.append(f"\\cmidrule(lr){{{current_col}-{end_col}}}")
                current_col = end_col + 1
            latex_lines.append(" ".join(cmidrules))
            
            model_name_map = {'gpt-oss-20b': 'gpt-20b', 'llama3-8b-instruct': 'llama3-8b', 'qwen3-32b-v1': 'qwen-32b'}
            col_header = r"\textbf{Injection Type}" if group_name == "Base" else r"\textbf{Language}"
            model_headers = [col_header] + [model_name_map.get(m, m).replace('_', '\\_') for m in models] * len(metrics)
            latex_lines.append(" & ".join(model_headers) + " \\\\")
            latex_lines.append(r"\midrule")
            
            for lang in langs:
                lang_val = lang.replace('_', '\\_')
                if lang_val == "raw":
                    lang_val = r"\textbf{\textit{no-injection}}"
                else:
                    if group_name == "First":
                        lang_val = lang_val.replace("\\_first", "")
                    elif group_name == "Last":
                        lang_val = lang_val.replace("\\_last", "")
                    if group_name == "Base":
                        lang_val = f"{lang_val}-qp random"
                        
                row_vals = [lang_val]
                for metric in metrics:
                    for model in models:
                        mask = (group_df['Language/Variant'] == lang) & (group_df['Model'] == model)
                        val_series = group_df.loc[mask, metric]
                        if not val_series.empty:
                            row_vals.append(f"{val_series.values[0]:.4f}")
                        else:
                            row_vals.append("-")
                latex_lines.append(" & ".join(row_vals) + " \\\\")
                
            latex_lines.append(r"\bottomrule")
            latex_lines.append(r"\end{tabular}")
            if year == "2021":
                latex_lines.append(r"\vspace{2em}")
                latex_lines.append("")
            
        latex_lines.append(r"\end{table}")
        
        latex_table = "\n".join(latex_lines)
        
        file_suffix = group_name.lower()
        output_path = output_dir / f"mae_report_{file_suffix}.tex"
        with open(output_path, "w", encoding="utf-8") as f:
            f.write(latex_table)
            
        print(f"LaTeX table ({group_name}) saved to {output_path.resolve()}\n")
        print(latex_table)
        print("\n")
else:
    print("No results to display.")

results_df


LaTeX table (Base) saved to D:\Work\Research_Project\anaconda_research_project\outputs\metrics_table\mae_report_base.tex

\begin{table}[t]
\centering
\caption{Agreement between LLM-generated relevance labels and NIST judgments for Base variants on TREC-DL 2021 and 2022.}
\label{tab:metrics_base}
\setlength{\tabcolsep}{3.5pt}
\footnotesize
\textbf{TREC-DL 2021} \\
\vspace{0.2em}
\begin{tabular}{lccccccccc}
\toprule
 & \multicolumn{3}{c}{\textbf{MAE}} & \multicolumn{3}{c}{\textbf{Mean-Diff}} & \multicolumn{3}{c}{\textbf{RMSE}} \\
\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10}
\textbf{Injection Type} & gpt-20b & llama3-8b & qwen-32b & gpt-20b & llama3-8b & qwen-32b & gpt-20b & llama3-8b & qwen-32b \\
\midrule
\textbf{\textit{no-injection}} & 0.7692 & 0.9729 & 0.8843 & 0.6106 & 0.7825 & 0.7833 & 1.1255 & 1.2596 & 1.2078 \\
ar-qp random & 0.8722 & 1.0317 & 1.0134 & 0.6960 & 0.8522 & 0.9415 & 1.1927 & 1.3162 & 1.3015 \\
eng-qp random & 0.8639 & 1.1608 & 1.0543 & 0.6643 & 1.0338 & 

,Group,Year,Model,Language/Variant,MAE,Mean-Diff,RMSE
0,First,2021,gpt-oss-20b,ar_first,0.997495,0.874739,1.291857
1,Base,2021,gpt-oss-20b,ar,0.872234,0.696033,1.192706
2,Last,2021,gpt-oss-20b,ar_last,0.917745,0.795825,1.228744
3,First,2021,gpt-oss-20b,eng_first,0.825887,0.655532,1.147567
4,Base,2021,gpt-oss-20b,eng,0.863883,0.664301,1.182863
...,...,...,...,...,...,...,...
199,Base,2022,qwen3-32b-v1,vi,0.952542,0.830508,1.225129
200,Last,2022,qwen3-32b-v1,vi_last,0.920527,0.790960,1.193832
201,First,2022,qwen3-32b-v1,zh_first,0.998493,0.896045,1.278388
202,Base,2022,qwen3-32b-v1,zh,0.994727,0.882486,1.254896


In [55]:
raw_df_all = results_df[results_df['Language/Variant'] == 'raw']
if not raw_df_all.empty:
    latex_lines = []
    latex_lines.append(r"\begin{table}[t]")
    latex_lines.append(r"\centering")
    latex_lines.append(r"\caption{Agreement between LLM-generated relevance labels and NIST judgments for the non-injected raw dataset on TREC-DL 2021 and 2022.}")
    latex_lines.append(r"\label{tab:metrics_raw}")
    latex_lines.append(r"\setlength{\tabcolsep}{3.5pt}")
    latex_lines.append(r"\footnotesize")
    
    for year in ["2021", "2022"]:
        year_df = raw_df_all[raw_df_all['Year'] == year]
        if year_df.empty:
            continue
            
        models = sorted(year_df['Model'].unique())
        model_name_map = {'gpt-oss-20b': 'gpt-20b', 'llama3-8b-instruct': 'llama3-8b', 'qwen3-32b-v1': 'qwen-32b'}
        display_models = [model_name_map.get(m, m).replace('_', '\_') for m in models]
        
        col_format = "l" + "c" * len(models)
        
        latex_lines.append(f"\\textbf{{TREC-DL {year}}} \\\\")
        latex_lines.append(r"\vspace{0.2em}")
        
        tabular_line = r"\begin{tabular}{" + col_format + r"}"
        latex_lines.append(tabular_line)
        latex_lines.append(r"\toprule")
        
        latex_lines.append(" & " + " & ".join([f"\\textbf{{{m}}}" for m in display_models]) + r" \\")
        latex_lines.append(r"\midrule")
        
        metrics = ['MAE', 'RMSE', 'Mean-Diff']
        for metric in metrics:
            row_vals = [f"\\textbf{{{metric}}}"]
            for model in models:
                mask = (year_df['Model'] == model)
                val_series = year_df.loc[mask, metric]
                if not val_series.empty:
                    row_vals.append(f"{val_series.values[0]:.4f}")
                else:
                    row_vals.append("-")
            latex_lines.append(" & ".join(row_vals) + r" \\")
            
        latex_lines.append(r"\bottomrule")
        latex_lines.append(r"\end{tabular}")
        
        if year == "2021":
            latex_lines.append(r"\vspace{2em}")
            latex_lines.append("")
            
    latex_lines.append(r"\end{table}")
    
    latex_table = "\n".join(latex_lines)
    
    output_path = output_dir / "mae_report_raw.tex"
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(latex_table)
        
    print(f"LaTeX table (Raw) saved to {output_path.resolve()}\n")
    print(latex_table)
    print("\n")


LaTeX table (Raw) saved to D:\Work\Research_Project\anaconda_research_project\outputs\metrics_table\mae_report_raw.tex

\begin{table}[t]
\centering
\caption{Agreement between LLM-generated relevance labels and NIST judgments for the non-injected raw dataset on TREC-DL 2021 and 2022.}
\label{tab:metrics_raw}
\setlength{\tabcolsep}{3.5pt}
\footnotesize
\textbf{TREC-DL 2021} \\
\vspace{0.2em}
\begin{tabular}{lccc}
\toprule
 & \textbf{gpt-20b} & \textbf{llama3-8b} & \textbf{qwen-32b} \\
\midrule
\textbf{MAE} & 0.7692 & 0.9729 & 0.8843 \\
\textbf{RMSE} & 1.1255 & 1.2596 & 1.2078 \\
\textbf{Mean-Diff} & 0.6106 & 0.7825 & 0.7833 \\
\bottomrule
\end{tabular}
\vspace{2em}

\textbf{TREC-DL 2022} \\
\vspace{0.2em}
\begin{tabular}{lccc}
\toprule
 & \textbf{gpt-20b} & \textbf{llama3-8b} & \textbf{qwen-32b} \\
\midrule
\textbf{MAE} & 0.6768 & 0.9751 & 0.8377 \\
\textbf{RMSE} & 1.0030 & 1.2456 & 1.1366 \\
\textbf{Mean-Diff} & 0.3906 & 0.7586 & 0.6614 \\
\bottomrule
\end{tabular}
\end{table}




<>:18: SyntaxWarning: invalid escape sequence '\_'
<>:18: SyntaxWarning: invalid escape sequence '\_'
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_20740\1381041161.py:18: SyntaxWarning: invalid escape sequence '\_'
  display_models = [model_name_map.get(m, m).replace('_', '\_') for m in models]
